## Second part (the Deployment)

In [4]:
from dotenv import load_dotenv
from lmstudio import get_lmstudio_client
import numpy as np

load_dotenv()

client = get_lmstudio_client()
MODEL_ID="text-embedding-embeddinggemma-300m-qat"

In [8]:
from sqlitesearch import VectorSearchIndex

vs_index = VectorSearchIndex(
    keyword_fields = ["course"],
    mode='ivf',
    db_path = "faq_vectors2.db",
)

In [19]:
query = 'I just discovered the course. Can I still join it?'
query_vector = np.array(client.embeddings.create(input=[query], model=MODEL_ID).data[0].embedding)

In [20]:
results = vs_index.search(query_vector, 
  filter_dict={"course": "llm-zoomcamp"},
  num_results=5)

In [21]:
results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': 'bd31146b0e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'When will the course be offered next?',
  'answer': 'Summer 2027.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': '69d122f12e',
  'course': 'l

### Now we are going to use RAG Vector class

In [22]:
from rag_vector import RAGVector

assistant = RAGVector(
    embedder = MODEL_ID,
    index = vs_index,
    llm_client = client)

In [23]:
query = 'I just found out about the program, can I still sign up?'
#here you have to load another model in LLM studio (meta-llama/Meta-Llama-3-8B-Instruct)
assistant.rag(query)

"Yes, you can still join the program. However, if you want to receive a certificate, you need to submit your project while they're still accepting submissions."

In [24]:
vs_index.close()